In [10]:
!pip install sentence-transformers faiss-cpu gradio transformers accelerate --quiet

print("✅ Light packages installed successfully")

✅ Light packages installed successfully


In [11]:
import gradio as gr
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

print("✅ Imports successful")

✅ Imports successful


In [12]:
# Embedding model for memory
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# FAISS vector store for long-term memory
dimension = 384
index = faiss.IndexFlatL2(dimension)
memory_texts = ["This is the starting memory of the assistant."]
memory_embeddings = embedder.encode(memory_texts)
index.add(memory_embeddings)

# LLM
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="float16",
    max_new_tokens=300
)

print("✅ Memory + LLM ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Memory + LLM ready!


In [19]:
def respond(message, history):
    if not message or len(message.strip()) < 2:
        return "Please type a proper message."

    try:
        # Search memory
        query_emb = embedder.encode([message])
        distances, indices = index.search(query_emb, k=3)

        relevant_memory = [memory_texts[i] for i in indices[0] if i < len(memory_texts)]
        context = "\n".join(relevant_memory)

        # Prompt with memory
        prompt = f"""You are a helpful AI Research Assistant.
Previous information from memory: {context}
User: {message}
Assistant:"""

        result = generator(prompt, temperature=0.7, do_sample=True)
        bot_message = result[0]['generated_text'].split("Assistant:")[-1].strip()

        # Save important info to memory
        if len(message) > 40:
            memory_texts.append(f"User: {message}\nAssistant: {bot_message}")
            new_emb = embedder.encode([memory_texts[-1]])
            index.add(new_emb)

        return bot_message

    except Exception as e:
        return f"Error: {str(e)[:120]}"

# Much more stable interface
demo = gr.ChatInterface(
    fn=respond,
    title="🚀 Project 4: AI Research Assistant with Memory",
    description="This assistant remembers important things you tell it using vector memory.",
    examples=[
        "Remember that my name is Z and I am building 20 GenAI projects.",
        "What do you remember about me?",
        "What project am I currently working on?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://989796b343a4ff76a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
